# L10 — Sensor Data, Jupyter Notebooks, and Visualization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yanluo/stem-on-stage-notebooks/blob/main/L10/L10_starter.ipynb)

**Goal:** load a recording captured from a micro:bit accelerometer, plot it, compute magnitude, smooth it, and find the peaks (jumps).

**Where to run this:** Google Colab (no install). Click the badge above, or *File → Open notebook → GitHub* in Colab. Runs locally too if you have JupyterLab — see the README in the course repo.

## Step 0 — Imports

All four libraries are pre-installed in Colab. Just run the cell.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("running in Colab" if IN_COLAB else "running locally")

## Step 1 — Load the sample CSV

We'll start with a bundled walking trace so you can see the whole pipeline before capturing your own data. The sample loads straight from the public course repo — no upload needed.

In [ ]:
SAMPLE_URL = "https://raw.githubusercontent.com/yanluo/stem-on-stage-notebooks/main/data/sample-walk.csv"

if IN_COLAB:
    df = pd.read_csv(SAMPLE_URL)
else:
    df = pd.read_csv("../data/sample-walk.csv")

print(df.shape)
df.head()

**Sanity check:** the columns should be `time (seconds)`, `x`, `y`, `z`. If your CSV has different names (older MakeCode versions sometimes used `t`), rename here or adjust the code below.

## Step 2 — Plot X, Y, Z over time

In [ ]:
t = df["time (seconds)"]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, df["x"], label="x")
ax.plot(t, df["y"], label="y")
ax.plot(t, df["z"], label="z")
ax.set_xlabel("time (s)")
ax.set_ylabel("acceleration (mg)")
ax.legend()
ax.grid(True)
plt.show()

## Step 3 — Compute magnitude

$|a| = \sqrt{x^2 + y^2 + z^2}$. At rest this is ~1000 mg (gravity).

In [ ]:
df["mag"] = np.sqrt(df["x"]**2 + df["y"]**2 + df["z"]**2)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df["time (seconds)"], df["mag"], color="purple")
ax.axhline(1000, linestyle="--", color="gray", label="rest")
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
ax.legend()
plt.show()

## Step 4 — Smooth the signal

A 5-sample rolling mean is enough to kill most accelerometer noise without blurring real motion.

In [ ]:
df["mag_smooth"] = df["mag"].rolling(window=5, center=True).mean()

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df["time (seconds)"], df["mag"],        alpha=0.3, label="raw")
ax.plot(df["time (seconds)"], df["mag_smooth"],            label="smoothed")
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
ax.legend()
plt.show()

## Step 5 — Find the peaks

Tune `height` (minimum peak value) and `distance` (minimum samples between peaks) until the red `x`'s land where you expect.

In [ ]:
peaks, _ = find_peaks(df["mag_smooth"].fillna(0),
                      height=1300,
                      distance=4)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df["time (seconds)"], df["mag_smooth"])
ax.plot(df["time (seconds)"].iloc[peaks],
        df["mag_smooth"].iloc[peaks], "rx", markersize=12)
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
plt.show()

print(f"Detected {len(peaks)} peaks")

## Your turn — analyze your own recording

1. **Capture** a 30-second recording in MakeCode (Chrome or Edge): *Show Console Device → Download*. See the L10 slides for details.
2. **Upload** the CSV using the cell below (Colab) — or drop it into the file browser if you're running locally.
3. **Re-run** Steps 2–5 with `df = pd.read_csv(MY_FILE)`.
4. Tune `height` and `distance` in Step 5 until the peak count matches what you actually did.
5. **Save your work:** *File → Download .ipynb* before the runtime times out.

In [ ]:
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()        # opens a file picker
    MY_FILE = next(iter(uploaded))
else:
    MY_FILE = "../data/microbit-data-2026-04-17T20-52-48-462Z.csv"

df = pd.read_csv(MY_FILE)
print(df.shape)
df.head()

**Write 2–3 sentences below** describing what feature distinguishes your move from a baseline (rest or walking).

*Your answer here.*